---
last_verified: 2026-10-04
tool_version: n/a
sources: []
---

# Choosing between DB and registry sources in Grype

> Reference notebook. Release-neutral by design: no Grype version is claimed here, and flag names should be taken from the installed binary (`grype --help`), not from memory. No external sources are cited because the current research cycle covers other tools; all cross-references below point at files already in this kit.

## Purpose

A Grype scan needs two inputs: something to scan and a vulnerability database to scan it against. Those two inputs can arrive through two very different paths, and picking the wrong one for an environment is a common source of slow, flaky, or silently stale pipelines. This notebook compares the two source shapes, gives a decision rule for choosing between them, and provides small runnable checks that encode the rule so a pipeline can apply it instead of re-deciding it every run.

The two shapes under comparison:

- **Registry-image source** — point Grype at an image reference (for example `grype <image>`). Grype pulls the image layers from the container registry and resolves packages from what it finds. This is the shape `grype/notes/2026-06-08-first-grype-scan.md` records from a first manual run.
- **DB-backed SBOM source** — resolve packages once (for example with Syft), keep the SBOM file, and scan it offline against a local vulnerability database (the `grype sbom:<file>` form with the database refreshed separately via `grype db update`). This is the shape `grype/docs/vulnerability-management-pipeline-integration.md` recommends for environments without egress.

## When to use this

- A pipeline scans the same artifact in more than one stage and each stage re-pulls the image.
- Scans run in an environment where registry egress is restricted, metered, or absent.
- Findings changed between two builds and nobody can tell whether the image, the database, or the scan target changed.
- A scheduled scan needs a defined answer to "how stale may the vulnerability database be before this result stops counting".

## Prerequisites

- `grype` on `PATH` for the real scans; confirm with `grype version` and take flag names from `grype --help`.
- Python 3.8+ with only the standard library (`json`, `datetime`, `pathlib`) for the cells below — every code cell runs offline against embedded fixtures.
- A place to keep the last accepted report, as described in `grype/docs/vulnerability-management-pipeline-integration.md`.

In [ ]:
import json
from datetime import date
from pathlib import Path

WORK_DIR = Path("grype-source-decision")
WORK_DIR.mkdir(exist_ok=True)
print(f"Working directory: {WORK_DIR.resolve()}")

## Step 1: name the trade-off

The two sources differ on four axes. Cost here means wall-clock time and network dependence per scan, not severity coverage — both shapes consult the same local vulnerability database once it is present, so the finding set for identical inputs should agree and any disagreement means the inputs were not actually identical (different image tag, different database build, or a different target form).

| Axis | Registry-image source | DB-backed SBOM source |
|---|---|---|
| Network per scan | pulls image layers every run | none after the SBOM and database are in place |
| Database handling | downloaded implicitly on first run, inside the scan's time budget | refreshed explicitly as its own step, then cached between runs |
| Repeat scans | full cost every time | cheap — re-scan the stored SBOM file |
| Works without egress | no | yes, when the database was updated elsewhere and carried in |
| Failure mode | registry unreachable means no scan at all | stale database means a scan that looks complete but misses recent advisories |

The decision rule used below: without registry reachability the SBOM shape is the only option; with reachability but repeated scans of the same artifact, the SBOM shape still wins on cost; the registry-image shape is for one-off scans where pulling the image is already part of the job.

In [ ]:
def choose_source(*, registry_reachable, db_cached, repeated_scans):
    """Return (source, rationale) for the given environment constraints."""
    if not registry_reachable:
        return (
            "db-backed-sbom",
            "No registry egress: scan the stored SBOM against a database that was "
            "refreshed elsewhere and carried in.",
        )
    if repeated_scans:
        return (
            "db-backed-sbom",
            "Same artifact scanned more than once: resolve packages once, re-scan "
            "the SBOM file instead of re-pulling layers.",
        )
    if not db_cached:
        return (
            "registry-image",
            "One-off scan with a reachable registry: scan the image reference "
            "directly and let the first run fetch the database.",
        )
    return (
        "db-backed-sbom",
        "Database already cached: the SBOM re-scan is the cheapest repeatable form.",
    )


# Scenario matrix: (registry_reachable, db_cached, repeated_scans) -> expected source.
scenarios = [
    ((False, False, False), "db-backed-sbom"),  # air-gapped: only option
    ((False, True, True), "db-backed-sbom"),  # air-gapped, warm cache
    ((True, False, False), "registry-image"),  # dev laptop, first scan
    ((True, True, True), "db-backed-sbom"),  # CI re-scanning one artifact
    ((True, True, False), "db-backed-sbom"),  # warm cache, single scan
]

for (reachable, cached, repeated), expected in scenarios:
    source, rationale = choose_source(
        registry_reachable=reachable, db_cached=cached, repeated_scans=repeated
    )
    status = "OK" if source == expected else "MISMATCH"
    print(f"[{status}] reachable={reachable} cached={cached} repeated={repeated}")
    print(f"         -> {source}: {rationale}")
    assert source == expected, f"decision drift for {(reachable, cached, repeated)}"

print(f"\nAll {len(scenarios)} scenarios match the decision rule.")

## Step 2: compare what each source reports

Both shapes ultimately produce the same report schema — a list of matches, each carrying the vulnerability identity and severity plus the affected package name and version (the same `.matches[]` fields the pipeline doc extracts with `jq`). The cell below works that schema with a small synthetic fixture so the counting logic can be reviewed without a registry, a database, or network access.

The fixture IDs are synthetic (`SYNTH-*`) and stand in for real advisory identifiers; they assert the shape of the counting code, not the content of any advisory.

In [ ]:
# Synthetic fixture in the report shape the kit's pipeline doc consumes:
# matches[].vulnerability.{id, severity} + matches[].artifact.{name, version}.
synthetic_report = {
    "matches": [
        {
            "vulnerability": {"id": "SYNTH-2026-0001", "severity": "High"},
            "artifact": {"name": "openssl", "version": "1.1.1"},
        },
        {
            "vulnerability": {"id": "SYNTH-2026-0002", "severity": "High"},
            "artifact": {"name": "curl", "version": "7.80.0"},
        },
        {
            "vulnerability": {"id": "SYNTH-2026-0003", "severity": "Medium"},
            "artifact": {"name": "zlib", "version": "1.2.11"},
        },
        {
            "vulnerability": {"id": "SYNTH-2026-0004", "severity": "Low"},
            "artifact": {"name": "expat", "version": "2.4.1"},
        },
    ],
    "source": {"target": "synthetic-fixture"},
}

fixture_path = WORK_DIR / "synthetic-grype-report.json"
fixture_path.write_text(json.dumps(synthetic_report, indent=2))


def severity_summary(report):
    """Count matches per severity, mirroring the kit's jq triage extraction."""
    summary = {}
    for match in report.get("matches", []):
        severity = match["vulnerability"]["severity"]
        summary[severity] = summary.get(severity, 0) + 1
    return summary


loaded = json.loads(fixture_path.read_text())
summary = severity_summary(loaded)
print(f"Matches: {len(loaded['matches'])}, by severity: {summary}")
assert summary == {"High": 2, "Medium": 1, "Low": 1}
assert len(loaded["matches"]) == sum(summary.values())
print("Fixture round-trips through the counting logic unchanged.")

## Step 3: gate on database freshness

The DB-backed shape has one failure mode the registry shape does not: a scan against an old database completes successfully and reports fewer findings, which reads as "clean" rather than "stale". The pipeline doc already says to refresh deliberately with `grype db update` as its own step; the remaining question is how stale is too stale. The cell below encodes that as a check — a maximum database age in days — so a scheduled job can fail loudly instead of passing quietly. The dates are fixed fixtures so the check is deterministic; a real job would substitute the database build date it recorded at refresh time.

In [ ]:
def db_is_fresh(db_build, evaluated_on, max_age_days):
    """True when the database build date is within the allowed age."""
    age_days = (evaluated_on - db_build).days
    return age_days <= max_age_days, age_days


evaluated_on = date(2026, 10, 4)
cases = [
    # (db build date, max age days, expected fresh?)
    (date(2026, 10, 3), 7, True),  # refreshed yesterday
    (date(2026, 9, 27), 7, True),  # exactly at the boundary
    (date(2026, 9, 20), 7, False),  # two weeks old: stale
    (date(2026, 9, 20), 30, True),  # same build, looser policy: fresh
]

for build, max_age, expected in cases:
    fresh, age = db_is_fresh(build, evaluated_on, max_age)
    status = "OK" if fresh == expected else "MISMATCH"
    print(f"[{status}] built={build} age={age}d max={max_age}d -> fresh={fresh}")
    assert fresh == expected, f"freshness drift for build {build}"

print(f"\nAll {len(cases)} freshness cases behave as specified.")

## Verify

- Run the notebook top to bottom from a clean checkout with no network: every code cell uses only the standard library and embedded fixtures, so all assertions should pass offline.
- Confirm the decision rule against a real environment by answering the three inputs (registry reachable? database cached? same artifact scanned repeatedly?) and checking the recommended shape matches what the pipeline already does.
- For a live cross-check, scan the same artifact once per shape with the installed binary and diff the finding sets; any disagreement means the inputs differed (image tag, database build, or target form), not the shapes.

## Common errors

- **Changing the target form between stages.** A stage that silently moves from an image reference to a directory scans something else and reports a smaller finding set that reads as an improvement. Pick one form per stage, as the pipeline integration doc advises.
- **Treating a stale-database scan as a clean scan.** An old database produces a complete-looking report with missing advisories. Keep the refresh as a separate step and enforce a maximum database age rather than assuming freshness.
- **Re-pulling in every stage.** Scanning the same image reference in build, test, and release stages pays the registry cost three times; resolve packages once and re-scan the stored SBOM.
- **Caching flag names from memory.** Take flag and subcommand names from the installed binary's own help output; an option remembered from another release may not exist in the one running the job.

## References

- `grype/docs/vulnerability-management-pipeline-integration.md` — the pipeline seams around target forms, explicit database refresh, and report retention.
- `grype/docs/grype-syft-integration-guide.md` — generating the SBOM that the DB-backed shape scans.
- `grype/notes/2026-06-08-first-grype-scan.md` — first-run notes including the initial database download.
- `grype/configs/grype-ci-github-actions.yaml` — the CI workflow config the report field set here is consistent with.
- `grype/notebooks/grype-sbom-output-explorer.ipynb` — the companion notebook on output formats (JSON, CycloneDX, SARIF) for whatever the chosen source produces.
- `grype/scripts/grype-pipeline-ci-scan.sh` — the runnable scan-and-gate script this decision feeds into.